# Introduction to LLMs

Mental model for large language models as next-token predictors: tokens, context windows, temperature, and how prompts steer inference.


## 1. Overview

This guide covers:

- How LLMs generate text by repeatedly predicting the next token
- Tokenization, context budgets, and why they affect cost and limits
- Temperature and decoding as controls on variability (not truth)
- Local inspection with `tiktoken` and a toy bigram generator
- A live Chat Completions call with response metadata inspection


## 2. Motivation

Production systems built on LLMs fail when teams treat them like deterministic databases. Billing, rate limits, and truncation are measured in **tokens**. Outputs can be fluent and wrong. Prompt engineering changes **inputs at inference time** — it does not retrain weights.

Before few-shot patterns, agents, or retrieval, you need a shared vocabulary: what the model is doing mathematically, what fits in context, and what temperature actually controls.


## 3. Concepts

### 3.1 Glossary

| Term | Meaning |
|------|--------|
| **LLM** | A neural network trained on large text corpora to model language |
| **Token** | A chunk of text (word piece, word, or punctuation) the model reads/writes |
| **Context window** | Maximum tokens in one request (input + output budget) |
| **Training** | Offline process that updates model weights from data |
| **Inference** | Runtime generation: prompt → model → completion |
| **Temperature** | Sampling randomness; lower ≈ more deterministic, higher ≈ more varied |
| **Hallucination** | Fluent but incorrect or invented content |
| **Prompt engineering** | Shaping inputs so likely continuations match your intent |

### 3.2 How it works

1. Text is split into **tokens** and embedded as vectors.
2. Transformer layers process the sequence and produce a distribution over the **next token**.
3. A decoding strategy (greedy, temperature sampling, top-p, etc.) selects the next token.
4. The chosen token is appended; steps 2–3 repeat until a stop condition (end token, max tokens, stop sequence).

Prompt engineering changes **input conditioning** so the highest-probability continuations align with your task.

### 3.3 When to use an LLM

**Good fit:** drafting, rewriting, classification with nuance, extraction, summarization, brainstorming, code assistance, conversational UX.

**Poor fit alone:** exact arithmetic without tools, guaranteed legal/medical advice, perfect recall of private data not in context, cryptographic correctness.

**Trade-offs:** fast to prototype and flexible, but non-deterministic, costly at scale, and requiring evaluation and safety controls.


## 4. Architecture

### Training vs inference

```mermaid
flowchart LR
    subgraph train [Training offline]
        data[Large text corpus] --> trainStep[Update weights]
        trainStep --> weights[Model weights]
    end
    subgraph infer [Inference online]
        prompt[Your prompt] --> model[Frozen LLM]
        model --> completion[Generated tokens]
    end
    weights -.-> model
```

### Token → predict → append loop

```text
Prompt:  "The capital of France is"
Tokens:  [The] [capital] [of] [France] [is]
                |
                v
        Model predicts next-token distribution
                |
                v
Sample:  "Paris"  →  append  →  continue until stop
```


## 5. Local Python Examples

Cost, rate limits, and context limits are measured in **tokens**, not characters. The cells below use `tiktoken` (no API) and a toy bigram model to illustrate generation shape.


In [1]:
# Local tokenization demo — no API key required
from __future__ import annotations

import tiktoken


def count_tokens(text: str, encoding_name: str = "o200k_base") -> int:
    # Return token count for text using the named tiktoken encoding.
    enc = tiktoken.get_encoding(encoding_name)
    return len(enc.encode(text))


encoding = tiktoken.get_encoding("o200k_base")

samples = [
    "Hello, world!",
    "Prompt engineering steers next-token prediction.",
    "def add(a, b):\n    return a + b",
    "The quick brown fox jumps over the lazy dog.",
]

print(f"{'Text':<50} {'Tokens':>6}  Token IDs (first 8)")
print("-" * 90)

for text in samples:
    token_ids = encoding.encode(text)
    pieces = [encoding.decode([tid]) for tid in token_ids]
    preview = text.replace("\n", "\\n")
    print(f"{preview:<50} {len(token_ids):>6}  {token_ids[:8]}")
    print(f"  pieces: {pieces}\n")


Text                                               Tokens  Token IDs (first 8)
------------------------------------------------------------------------------------------
Hello, world!                                           4  [13225, 11, 2375, 0]
  pieces: ['Hello', ',', ' world', '!']

Prompt engineering steers next-token prediction.        8  [51905, 16411, 2310, 409, 2613, 73397, 35611, 13]
  pieces: ['Prompt', ' engineering', ' ste', 'ers', ' next', '-token', ' prediction', '.']

def add(a, b):\n    return a + b                       11  [1314, 1147, 6271, 11, 287, 1883, 271, 622]
  pieces: ['def', ' add', '(a', ',', ' b', '):\n', '   ', ' return', ' a', ' +', ' b']

The quick brown fox jumps over the lazy dog.           10  [976, 4853, 19705, 68347, 65613, 1072, 290, 29082]
  pieces: ['The', ' quick', ' brown', ' fox', ' jumps', ' over', ' the', ' lazy', ' dog', '.']



### 5.1 Token density comparison


In [2]:
# Compare token density across plain text, jargon, and code
examples = {
    "plain": "Large language models predict the next token.",
    "jargon": (
        "Autoregressive transformers approximate conditional token "
        "distributions via self-attention."
    ),
    "code": "def greet(name):\n    return f'Hello, {name}!'\n\nprint(greet('Ada'))\n",
}

print(f"{'label':<8} {'tokens':>6} {'chars':>6} {'tok/char':>10}")
for label, text in examples.items():
    n_tok = count_tokens(text)
    n_chr = len(text)
    density = n_tok / n_chr if n_chr else 0.0
    print(f"{label:<8} {n_tok:>6} {n_chr:>6} {density:>10.3f}")


label    tokens  chars   tok/char
plain         8     45      0.178
jargon       13     91      0.143
code         20     67      0.299


### 5.2 Toy next-token simulator

A hand-built bigram table clarifies the *shape* of generation (condition → sample → append), not production quality.


In [3]:
# Toy bigram next-token generator — no API required
from __future__ import annotations

import random
from typing import Dict, List

BIGRAMS: Dict[str, List[str]] = {
    "the": ["cat", "dog", "model"],
    "cat": ["sat", "ate"],
    "dog": ["ran", "barked"],
    "model": ["predicts", "learns"],
    "sat": ["on"],
    "on": ["the"],
    "predicts": ["tokens", "text"],
    "tokens": ["one", "step"],
    "one": ["by"],
    "by": ["one"],
    "text": ["."],
    "learns": ["patterns"],
    "patterns": ["."],
    "ate": ["food"],
    "food": ["."],
    "ran": ["away"],
    "away": ["."],
    "barked": ["loudly"],
    "loudly": ["."],
    "step": ["."],
}


def toy_generate(seed: str, max_words: int = 12, rng_seed: int = 42) -> str:
    # Generate a short phrase by sampling from BIGRAMS.
    random.seed(rng_seed)
    words = [seed.lower()]
    for _ in range(max_words):
        last = words[-1]
        options = BIGRAMS.get(last)
        if not options:
            break
        nxt = random.choice(options)
        words.append(nxt)
        if nxt == ".":
            break
    return " ".join(words)


for start in ("the", "model", "cat"):
    print(f"seed={start!r:8} -> {toy_generate(start)}")


seed='the'    -> the model predicts tokens step .
seed='model'  -> model predicts tokens step .
seed='cat'    -> cat sat on the cat sat on the model predicts text .


### 5.3 Context budget check

Estimate whether a long prompt fits a fictional 8k context when reserving tokens for the completion.


In [4]:
# Context budget estimation with tiktoken
CONTEXT_LIMIT = 8000
RESERVED_COMPLETION = 1000
AVAILABLE_FOR_PROMPT = CONTEXT_LIMIT - RESERVED_COMPLETION

paragraph = (
    "Customer emails arrive all day. An LLM can draft replies, summarize threads, "
    "and suggest next actions for human agents. "
)
long_prompt = paragraph * 120

prompt_tokens = count_tokens(long_prompt)
remaining = AVAILABLE_FOR_PROMPT - prompt_tokens
fits = remaining >= 0

print(f"prompt_tokens={prompt_tokens}")
print(f"available_for_prompt={AVAILABLE_FOR_PROMPT}")
print(f"remaining={remaining}")
print(f"fits={fits}")


prompt_tokens=2881
available_for_prompt=7000
remaining=4119
fits=True


## 6. OpenAI SDK Examples

Standard client bootstrap (see `Environment_Setup.ipynb` for venv and `.env` setup):

```python
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(root / ".env")
client = OpenAI()
```


In [5]:
# Chat completion with response inspection
from __future__ import annotations

import os
from pathlib import Path

from dotenv import load_dotenv
from openai import (
    APIConnectionError,
    APIError,
    AuthenticationError,
    OpenAI,
    RateLimitError,
)

def find_project_root(start: Path | None = None) -> Path:
    """Walk upward until requirements.txt is found (project root)."""
    here = (start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find requirements.txt. Open the notebook from this repository "
        "or set the working directory to the project root."
    )

root = find_project_root()
load_dotenv(root / ".env")
client = OpenAI()

MODEL = "gpt-4o-mini"


def api_key_ready() -> bool:
    key = os.getenv("OPENAI_API_KEY", "")
    if not key.strip():
        print("OPENAI_API_KEY not configured. Copy .env.example to .env and set your key.")
        return False
    if "your_openai_api_key" in key.lower():
        print("OPENAI_API_KEY is still a placeholder. Replace it with a real key.")
        return False
    return True


def chat_once(
    user_prompt: str,
    *,
    system: str = "You are a clear, concise technical writer.",
    temperature: float = 0.2,
) -> str | None:
    # Send one Chat Completions request and return assistant text.
    if not api_key_ready():
        return None

    try:
        response = client.chat.completions.create(
            model=MODEL,
            temperature=temperature,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": user_prompt},
            ],
        )
    except AuthenticationError:
        print("Authentication failed — check OPENAI_API_KEY.")
        return None
    except RateLimitError:
        print("Rate limited — wait and retry.")
        return None
    except APIConnectionError as exc:
        print(f"Network error: {exc}")
        return None
    except APIError as exc:
        print(f"OpenAI API error: {exc}")
        return None

    choice = response.choices[0]
    usage = response.usage
    print(f"model={response.model}  finish={choice.finish_reason}")
    if usage:
        print(
            f"tokens: prompt={usage.prompt_tokens}  "
            f"completion={usage.completion_tokens}  "
            f"total={usage.total_tokens}"
        )
    return choice.message.content or ""


answer = chat_once(
    "In 3 short bullet points, explain what an LLM is and why prompts matter."
)
if answer:
    print("\n--- Assistant ---\n")
    print(answer)


model=gpt-4o-mini-2024-07-18  finish=stop
tokens: prompt=38  completion=105  total=143

--- Assistant ---

- **Definition**: A Large Language Model (LLM) is an advanced AI system designed to understand and generate human-like text based on vast amounts of training data.

- **Functionality**: LLMs utilize complex algorithms to predict and produce coherent text, enabling applications in chatbots, content creation, and more.

- **Importance of Prompts**: Prompts guide the LLM's responses; well-crafted prompts can elicit more relevant, accurate, and contextually appropriate outputs, enhancing user interaction and satisfaction.


### 6.1 Temperature comparison


In [6]:
# Same prompt, different temperature — observe variability
prompt = "Give one creative analogy for how an LLM generates text. One sentence only."

for temp in (0.0, 0.8):
    print(f"\n=== temperature={temp} ===")
    result = chat_once(prompt, temperature=temp)
    if result:
        print(result)



=== temperature=0.0 ===


model=gpt-4o-mini-2024-07-18  finish=stop
tokens: prompt=36  completion=32  total=68
An LLM generates text like a skilled chef crafting a dish, blending ingredients of language and context to create a unique recipe of words that tantalizes the mind.

=== temperature=0.8 ===


model=gpt-4o-mini-2024-07-18  finish=stop
tokens: prompt=36  completion=35  total=71
An LLM generates text like a skilled chef blending ingredients from a vast pantry to create a unique dish, carefully balancing flavors based on the recipe of language patterns it has learned.


### 6.2 Vague vs specific prompts


In [7]:
# Prompt clarity affects structure and usefulness
vague = "Tell me about LLMs in customer support."
specific = (
    "You are briefing a support team lead.\n"
    "List exactly 4 benefits of using LLMs in customer support.\n"
    "Use a numbered list. Each item: one bold benefit title, then one sentence.\n"
    "Avoid hype; mention one risk in a final sentence."
)

print("=== VAGUE ===")
vague_out = chat_once(vague)
if vague_out:
    print(vague_out[:600], "..." if len(vague_out) > 600 else "")

print("\n=== SPECIFIC ===")
specific_out = chat_once(specific)
if specific_out:
    print(specific_out)


=== VAGUE ===


model=gpt-4o-mini-2024-07-18  finish=stop
tokens: prompt=30  completion=371  total=401
Large Language Models (LLMs) are increasingly being utilized in customer support to enhance efficiency, improve response times, and provide personalized assistance. Here are some key aspects of their application in this field:

### 1. **Automated Responses**
LLMs can generate instant replies to common customer inquiries, reducing wait times and freeing up human agents for more complex issues.

### 2. **24/7 Availability**
LLMs enable round-the-clock support, allowing customers to receive assistance outside of regular business hours.

### 3. **Personalization**
By analyzing customer data and pr ...

=== SPECIFIC ===


model=gpt-4o-mini-2024-07-18  finish=stop
tokens: prompt=70  completion=139  total=209
1. **24/7 Availability**  
LLMs can provide round-the-clock support, ensuring customers receive assistance at any time without delays.

2. **Scalability**  
These models can handle a large volume of inquiries simultaneously, allowing support teams to manage peak times without additional staffing.

3. **Consistency in Responses**  
LLMs deliver uniform answers to common questions, reducing the likelihood of conflicting information being provided to customers.

4. **Cost Efficiency**  
Implementing LLMs can lower operational costs by automating routine inquiries, freeing up human agents for more complex issues.

However, there is a risk of LLMs misunderstanding nuanced customer queries, potentially leading to unsatisfactory resolutions.


## 7. Implementation notes

1. **`find_project_root`** — Notebooks often run with cwd = `notebooks/`. Walking up for `requirements.txt` loads `.env` from the repo root.
2. **`tiktoken`** — Encodes strings offline so you can estimate cost and context usage before calling the API.
3. **Toy bigram model** — Demonstrates the generate loop; real models use billions of parameters and subword tokens.
4. **`chat_once`** — Wraps `client.chat.completions.create` with typed errors and skips the call when the key is missing.
5. **`response.usage`** — Reports prompt/completion token counts for observability and budgeting.
6. **Temperature** — Controls sampling randomness, not factual accuracy.


## 8. Best practices

- Treat prompts as **product code**: version, test, and review them.
- Measure **tokens** early; long prompts cost more and can truncate context.
- Prefer **clear instructions** over vague requests ("list 3 bullets" beats "tell me about LLMs").
- Separate **system** (policy/persona) from **user** (task/data) messages.
- Assume outputs can be wrong; add evaluation, grounding, or human review for high-stakes use.
- Start with a capable economical model (e.g. `gpt-4o-mini`) while iterating on prompts.


## 9. Common failure modes

| Symptom | Likely cause | Fix |
|---------|--------------|-----|
| Unexpected API cost | Prompt measured in characters, billed in tokens | Use `tiktoken` to estimate before sending |
| Truncated or incomplete answer | Input + output exceeds context window | Summarize, retrieve, or split the task |
| Different text on every run | Temperature > 0 or sampling enabled | Lower temperature for stability; log seeds if needed |
| Confident but wrong facts | Hallucination under weak grounding | Add sources, tools, or verification steps |
| "Training" the model via prompts | Confusing training with inference | Prompting changes inputs only; weights stay frozen |
| Secrets in notebooks | Key pasted into cells or outputs | Keep keys in `.env`; never print `OPENAI_API_KEY` |


## 10. Validation checklist

1. Run the tokenization cell and confirm token pieces differ from whole words.
2. Run the density comparison and note that code is often denser (more tokens per character).
3. Run the context budget cell and verify `fits` logic for your sample length.
4. Run `chat_once` with a valid key and confirm `finish_reason`, model id, and token usage print.
5. Compare vague vs specific prompts and confirm the specific output matches the requested structure.


## 11. Summary

- LLMs generate text by repeatedly predicting the **next token** given prior context.
- **Tokens**, **context windows**, and **temperature** shape cost, length, and variability.
- Prompt engineering steers inference without retraining the model.
- Local `tiktoken` inspection and a minimal Chat Completions call are enough to validate your setup.

**Next:** `OpenAI_SDK.ipynb` — client initialization, message roles, response fields, and error handling.
